In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (precision_score, recall_score,
                             average_precision_score,
                             accuracy_score, precision_recall_curve,
                             f1_score)

In [2]:
RANDOM_SEED = 42
df = pd.read_csv('../data/Synthetic_Financial_datasets_log.csv')

In [3]:
leakage_safe_features = ['step', 'type', 'amount']

In [4]:
def data_split(
        new_feature_column=None,
        data=df,
        base_features=['step', 'type', 'amount', 'isFraud']
):
    temp_df = data.copy()

    if new_feature_column is not None:
        base_features.append(new_feature_column)

    temp_df = temp_df[base_features]

    test_point = 550
    calibration_point = test_point - 50
    validation_point = calibration_point - 100

    validation = temp_df[(temp_df['step'] <= calibration_point) & (temp_df['step'] > validation_point)].copy()
    train = temp_df[temp_df['step'] <= validation_point].copy()

    X_train, y_train = train.drop('isFraud', axis=1), train['isFraud']
    X_val, y_val = validation.drop('isFraud', axis=1), validation['isFraud']

    return X_train, y_train, X_val, y_val

def model_estimation(y_true, y_proba):
    y_pred = (y_proba >= 0.5).astype(int)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    average_precision = average_precision_score(y_true, y_proba)

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'average_precision': average_precision,
    }

## Base Models

In [5]:
def get_preprocessor(
        numeric_features:List[str],
        categorical_features:List[str],
        other_features:List[str]
):
    preprocessor = ColumnTransformer(transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(
            drop='first',
            sparse_output=False,
            handle_unknown='ignore'
        ), categorical_features),
        ('keep', 'passthrough', other_features)
    ], remainder='drop')

    return preprocessor


In [6]:
def get_proba_predictions(
        X_train,
        y_train,
        X_val,
        numeric_features:List[str],
        categorical_features:List[str],
        other_features:List[str]=[]
):

    preprocessor = get_preprocessor(
        numeric_features,
        categorical_features,
        other_features
    )
    logreg_unbalanced = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight=None,
                                     random_state=RANDOM_SEED))
    ])

    logreg_balanced = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight='balanced',
                                     random_state=RANDOM_SEED))
    ])

    logreg_unbalanced.fit(X_train, y_train)
    logreg_balanced.fit(X_train, y_train)

    y_proba_unbalanced = logreg_unbalanced.predict_proba(X_val)[:, 1]
    y_proba_balanced = logreg_balanced.predict_proba(X_val)[:, 1]

    return y_proba_unbalanced, y_proba_balanced


In [7]:
unbalanced_res_table = pd.DataFrame(columns=[
    'Model',
    'precision',
    'recall',
    'f1',
    'accuracy',
    'average_precision',
])

balanced_res_table = pd.DataFrame(columns=[
    'Model',
    'precision',
    'recall',
    'f1',
    'accuracy',
    'average_precision',
])

## Base Models Estimation

In [8]:
X_train, y_train, X_val, y_val = data_split()
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'unbalanced_base'
metrics_balanced['Model'] = 'balanced_base'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Logarithmic amount

In [9]:
df['log_amount'] = np.log1p(df['amount'])

In [10]:
X_train, y_train, X_val, y_val = data_split(new_feature_column='log_amount')
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount', 'log_amount'],
    categorical_features=['type']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'unbalanced_log_amount'
metrics_balanced['Model'] = 'balanced_log_amount'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Hour

In [11]:
df['hour'] = (df['step'] - 1) % 24

X_train, y_train, X_val, y_val = data_split(new_feature_column='hour')
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type'],
    other_features=['hour']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'unbalanced_hour'
metrics_balanced['Model'] = 'balanced_hour'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Day

In [12]:
df['day'] = (df['step'] - 1) // 24

X_train, y_train, X_val, y_val = data_split(new_feature_column='day')
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type'],
    other_features=['day']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'unbalanced_day'
metrics_balanced['Model'] = 'balanced_day'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Dest type

In [13]:
df['dest_type'] = df['nameDest'].str[0]

X_train, y_train, X_val, y_val = data_split(new_feature_column='dest_type')
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type', 'dest_type']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'unbalanced_dest_type'
metrics_balanced['Model'] = 'balanced_dest_type'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

In [14]:
unbalanced_res_table

,Model,precision,recall,f1,accuracy,average_precision
0,unbalanced_base,0.0,0.0,0.0,0.996055,0.06037
1,unbalanced_log_amount,0.0,0.0,0.0,0.996055,0.093989
2,unbalanced_hour,0.0,0.0,0.0,0.996055,0.194985
3,unbalanced_day,0.0,0.0,0.0,0.996055,0.028323
4,unbalanced_dest_type,0.0,0.0,0.0,0.996055,0.05404


In [15]:
balanced_res_table

,Model,precision,recall,f1,accuracy,average_precision
0,balanced_base,0.032614,0.733395,0.062451,0.91313,0.107476
1,balanced_log_amount,0.032804,0.72417,0.062765,0.91468,0.109994
2,balanced_hour,0.034706,0.817343,0.066584,0.909596,0.147237
3,balanced_day,0.034718,0.817343,0.066607,0.909629,0.147214
4,balanced_dest_type,0.032602,0.733395,0.062429,0.913097,0.107467
